### Display the results

In [33]:
import glob
import pandas as pd
import os

# CONFIGURATION
BASE_PATH = "/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/logs/MSS_MSI/HISTOLAB/STANFORD_793"  
MODEL_NAMES = ['MAMBA_MIL']
METRICS = ['val_acc', 'val_bacc', 'val_macro_auc', 'val_macro_f1']

print("Model\tFold\t" + "\t".join(METRICS))

for model in MODEL_NAMES:
    print(f"--- {model} ---")
    for fold in range(1, 6):
        pattern = os.path.join(BASE_PATH, model, "resnet50_1024", "*", 
                               f"fold_{fold}", f"Best_Log_*{model}.csv")
        files = glob.glob(pattern)
        
        if files:
            try:
                df = pd.read_csv(files[0])
                # Multiply by 100 inside the f-string
                vals = [f"{df.iloc[-1].get(m, 0) * 100:.2f}" for m in METRICS]
                print(f"{fold}\t" + "\t".join(vals))
            except Exception:
                print(f"{fold}\tError")
        else:
            print(f"{fold}\tNot found")

Model	Fold	val_acc	val_bacc	val_macro_auc	val_macro_f1
--- IB_MIL ---
1	85.43	61.09	82.20	63.54
2	Not found
3	Not found
4	Not found
5	Not found
--- MEAN_MIL ---
1	86.09	56.13	84.65	57.29
2	Not found
3	Not found
4	Not found
5	Not found


### Save the results of specific parameters for all the models (MULTIPLE FOLDS)

In [45]:
import glob
import pandas as pd
import os

# CONFIGURATION
# CLAM, HISTOLAB, MUFASA, TRIDENT
PRE_PROCESSOR = "TRIDENT"
DATASET = "STANFORD_793"

BASE_PATH = f"/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/logs/MSS_MSI/{PRE_PROCESSOR}/{DATASET}/"  
MODEL_NAMES = ['AB_MIL', 'AC_MIL', 'ADD_MIL', 'CLAM_MB_MIL', 'DeepAttn_MIL', 'DGR_MIL', 'DS_MIL', 'DTFD_MIL','IB_MIL','ILRA_MIL','MAMBA_MIL',
                'MEAN_MIL', 'MHIM_MIL','MICO_MIL','PTC_MIL','TDA_MIL','TRANS_MIL'] 
# MODEL_NAMES = ['AB_MIL', 'AC_MIL', 'ADD_MIL', 'CLAM_MB_MIL', 'DeepAttn_MIL', 'DGR_MIL', 'DS_MIL', 'DTFD_MIL','IB_MIL','ILRA_MIL','MAMBA_MIL',
#                'MAMBA2D_MIL', 'MEAN_MIL', 'MHIM_MIL','MICO_MIL','PTC_MIL','TDA_MIL','TRANS_MIL'] 

# Check the parameters in the output file
METRICS = ['val_acc', 'val_bacc', 'val_macro_auc', 'val_macro_f1']
OUTPUT_FILE = f"{DATASET}/{PRE_PROCESSOR}_model_comparison_results.csv"

# List to store all records
all_results = []

for model in MODEL_NAMES:
    print(f"Processing {model}...")
    for fold in range(1, 6):
        # Find the specific CSV file for this model and fold
        # Pattern matches: .../ModelName/resnet50_1024/*/fold_X/Best_Log_*_ModelName.csv
        pattern = os.path.join(BASE_PATH, model, "resnet50_1024", "*", 
                               f"fold_{fold}", f"Best_Log_*{model}.csv")
        files = glob.glob(pattern)
        
        if files:
            try:
                # Read the CSV
                df = pd.read_csv(files[0])
                
                # Get the last row (best epoch)
                best_epoch = df.iloc[-1]
                
                # Create a dictionary for this row
                record = {
                    "Model": model,
                    "Fold": fold
                }
                
                # Extract metrics, multiply by 100, and round to 2 decimals
                for m in METRICS:
                    val = best_epoch.get(m, 0)
                    record[m] = round(val * 100, 2)
                
                all_results.append(record)
                
            except Exception as e:
                print(f"  Error reading Fold {fold}: {e}")
        else:
            print(f"  Fold {fold} file not found.")

# Create DataFrame and Save
if all_results:
    final_df = pd.DataFrame(all_results)
    
    # Optional: Sort by Model and Fold
    final_df = final_df.sort_values(by=["Model", "Fold"])

    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
        print(f"Created directory: {output_dir}")
    
    # Save to CSV
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Successfully saved results to: {OUTPUT_FILE}")
    print(final_df) # Print preview
else:
    print("\n❌ No data found to save.")

Processing AB_MIL...
Processing AC_MIL...
Processing ADD_MIL...
Processing CLAM_MB_MIL...
Processing DeepAttn_MIL...
Processing DGR_MIL...
Processing DS_MIL...
Processing DTFD_MIL...
Processing IB_MIL...
Processing ILRA_MIL...
Processing MAMBA_MIL...
Processing MEAN_MIL...
Processing MHIM_MIL...
Processing MICO_MIL...
Processing PTC_MIL...
Processing TDA_MIL...
Processing TRANS_MIL...

✅ Successfully saved results to: STANFORD_793/TRIDENT_model_comparison_results.csv
        Model  Fold  val_acc  val_bacc  val_macro_auc  val_macro_f1
0      AB_MIL     1    84.35     60.84          80.84         62.70
1      AB_MIL     2    86.67     67.20          79.15         69.87
2      AB_MIL     3    84.93     59.24          85.61         61.39
3      AB_MIL     4    85.91     63.55          76.41         66.01
4      AB_MIL     5    87.07     64.68          85.68         67.57
..        ...   ...      ...       ...            ...           ...
80  TRANS_MIL     1    85.03     53.75          79.3

### Save the results of specific parameters for all the models (MULTIPLE RUNS)

In [32]:
import glob
import pandas as pd
import os

# CONFIGURATION
# CLAM, HISTOLAB, MUFASA, TRIDENT
PRE_PROCESSOR = "MUFASA"
BASE_PATH = f"/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/logs/CLASSIFICATION/{PRE_PROCESSOR}/CAMELYON16/"  
MODEL_NAMES = ['AB_MIL', 'AC_MIL', 'ADD_MIL', 'CLAM_MB_MIL', 'DeepAttn_MIL', 'DGR_MIL', 'DS_MIL', 'DTFD_MIL','IB_MIL','ILRA_MIL','MAMBA_MIL',
                'MAMBA2D_MIL', 'MEAN_MIL', 'MHIM_MIL','MICO_MIL','PTC_MIL','TDA_MIL','TRANS_MIL'] 
# MODEL_NAMES = ['AB_MIL', 'AC_MIL', 'ADD_MIL', 'CLAM_MB_MIL', 'DeepAttn_MIL', 'DGR_MIL', 'DS_MIL', 'DTFD_MIL','IB_MIL','ILRA_MIL','MAMBA_MIL',
#                'MAMBA2D_MIL', 'MEAN_MIL', 'MHIM_MIL','MICO_MIL','PTC_MIL','TDA_MIL','TRANS_MIL'] 

# Check the parameters in the output file
METRICS = ['val_acc', 'val_bacc', 'val_macro_auc', 'val_macro_f1']
OUTPUT_FILE = f"CAMELYON16/{PRE_PROCESSOR}_model_comparison_results.csv"

# List to store all records
all_results = []

for model in MODEL_NAMES:
    print(f"Processing {model}...")
    for run in [41,1337,2023,3407,9999]:
        # Find the specific CSV file for this model and run
        # Pattern matches: .../ModelName/resnet50_1024/*/run_X/Best_Log_*_ModelName.csv
        pattern = os.path.join(BASE_PATH, model, "resnet50_1024", f"*seed_{run}*", "fold_*", f"Best_Log_seed{run}_*{model}.csv")
        files = glob.glob(pattern)
        if files:
            try:
                # Read the CSV
                df = pd.read_csv(files[0])
                
                # Get the last row (best epoch)
                best_epoch = df.iloc[-1]
                
                # Create a dictionary for this row
                record = {
                    "Model": model,
                    "Run": run
                }
                
                # Extract metrics, multiply by 100, and round to 2 decimals
                for m in METRICS:
                    val = best_epoch.get(m, 0)
                    record[m] = round(val * 100, 2)
                
                all_results.append(record)
                
            except Exception as e:
                print(f"  Error reading run {run}: {e}")
        else:
            print(f"  run {run} file not found.")

# Create DataFrame and Save
if all_results:
    final_df = pd.DataFrame(all_results)
    
    # Optional: Sort by Model and Run
    final_df = final_df.sort_values(by=["Model", "Run"])

    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
        print(f"Created directory: {output_dir}")
    
    # Save to CSV
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Successfully saved results to: {OUTPUT_FILE}")
    print(final_df) # Print preview
else:
    print("\n❌ No data found to save.")

Processing AB_MIL...
Processing AC_MIL...
Processing ADD_MIL...
Processing CLAM_MB_MIL...
Processing DeepAttn_MIL...
Processing DGR_MIL...
Processing DS_MIL...
Processing DTFD_MIL...
Processing IB_MIL...
Processing ILRA_MIL...
Processing MAMBA_MIL...
Processing MAMBA2D_MIL...
Processing MEAN_MIL...
Processing MHIM_MIL...
Processing MICO_MIL...
Processing PTC_MIL...
Processing TDA_MIL...
Processing TRANS_MIL...

✅ Successfully saved results to: CAMELYON16/MUFASA_model_comparison_results.csv
        Model   Run  val_acc  val_bacc  val_macro_auc  val_macro_f1
0      AB_MIL    41    79.07     76.40          80.74         77.08
1      AB_MIL  1337    80.62     77.65          79.44         78.56
2      AB_MIL  2023    80.62     77.26          80.13         78.32
3      AB_MIL  3407    74.42     73.44          80.26         73.15
4      AB_MIL  9999    85.27     80.61          80.20         82.67
..        ...   ...      ...       ...            ...           ...
85  TRANS_MIL    41    83.72 